## importar librerias

In [4]:
import pandas as pd
import numpy as np
import unicodedata
from sklearn.preprocessing import MinMaxScaler

## explorar dataset

In [5]:
df = pd.read_csv('datos_para_limpiar.csv')

# Visualizar las primeras 5 filas
display(df.head())

# Ver tipos de datos e información general
df.info()

# Conteo de valores nulos por columna
print("\nValores nulos por columna:")
print(df.isnull().sum())

,ID,Nombre,Edad,Salario,Ciudad,Genero,Nivel,Fecha,Ingresos,Gastos,Codigo,Notas
0,1,Ana,NaN,3.955413e+03,bogota,Masculino,Bajo,2018-12-29,4030.026008,1213.936751,ABC12345,Pendiente evaluación
1,2,Maria,NaN,3.054891e+03,NaN,Femenino,alto,2011-07-16,7509.810584,6038.896355,NaN,Buen empleado
2,3,Carlos,NaN,2.928345e+08,bogota,Femenino,Bajo,2022-04-07,9106.362328,6705.724812,NaN,NaN
3,4,Maria,11.0,4.718402e+03,NaN,NaN,medio,2022-10-11,NaN,NaN,XYZ98765,NaN
4,5,Laura,94.0,7.477876e+07,cali,masculino,alto,2017-07-11,3102.167244,7570.107277,NaN,Pendiente evaluación


<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 12 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   ID        100 non-null    int64  
 1   Nombre    100 non-null    str    
 2   Edad      50 non-null     float64
 3   Salario   65 non-null     float64
 4   Ciudad    82 non-null     str    
 5   Genero    73 non-null     str    
 6   Nivel     73 non-null     str    
 7   Fecha     100 non-null    str    
 8   Ingresos  48 non-null     float64
 9   Gastos    53 non-null     float64
 10  Codigo    69 non-null     str    
 11  Notas     67 non-null     str    
dtypes: float64(4), int64(1), str(7)
memory usage: 9.5 KB

Valores nulos por columna:
ID           0
Nombre       0
Edad        50
Salario     35
Ciudad      18
Genero      27
Nivel       27
Fecha        0
Ingresos    52
Gastos      47
Codigo      31
Notas       33
dtype: int64


## eliminar columnas 

In [6]:
df = df.drop(columns=['ID', 'Notas'])
df.head(2)

,Nombre,Edad,Salario,Ciudad,Genero,Nivel,Fecha,Ingresos,Gastos,Codigo
0,Ana,NaN,3955.412575,bogota,Masculino,Bajo,2018-12-29,4030.026008,1213.936751,ABC12345
1,Maria,NaN,3054.891167,NaN,Femenino,alto,2011-07-16,7509.810584,6038.896355,NaN


## normalizar nombres

In [7]:
def limpiar_nombre_columna(col):
    col = col.lower().strip()
    col = unicodedata.normalize('NFD', col).encode('ascii', 'ignore').decode('utf-8')
    col = col.replace(' ', '_')
    return col

df.columns = [limpiar_nombre_columna(c) for c in df.columns]
print("Nuevos nombres de columnas:", df.columns.tolist())

Nuevos nombres de columnas: ['nombre', 'edad', 'salario', 'ciudad', 'genero', 'nivel', 'fecha', 'ingresos', 'gastos', 'codigo']


## errores de escritura

In [8]:
def normalizar_ciudad(val):
    if pd.isna(val):
        return np.nan
    v = unicodedata.normalize('NFD', str(val).lower().strip()).encode('ascii', 'ignore').decode('utf-8')
    if 'bogota' in v:
        return 'Bogotá'
    elif 'medellin' in v:
        return 'Medellín'
    elif 'cali' in v:
        return 'Cali'
    return val

df['ciudad'] = df['ciudad'].apply(normalizar_ciudad)

# Limpieza ligera previa en género para evitar conflictos en la moda
df['genero'] = df['genero'].astype(str).str.strip().str.capitalize()
df['genero'] = df['genero'].replace({'Nan': np.nan, 'None': np.nan})

df['ciudad'].value_counts(dropna=False)

ciudad
Bogotá      41
Cali        21
Medellín    20
NaN         18
Name: count, dtype: int64

## valores nulos tratamiento

In [ ]:
# Convertir edad a numérico para calcular la media
df['edad'] = pd.to_numeric(df['edad'], errors='coerce')

# Media de edades válidas (0 a 120 años)
media_edad_valida = df[(df['edad'] >= 0) & (df['edad'] <= 120)]['edad'].mean()
df['edad'] = df['edad'].fillna(media_edad_valida)

# Mediana de salario
mediana_salario = df['salario'].median()
df['salario'] = df['salario'].fillna(mediana_salario)

# Moda de ciudad y género
moda_ciudad = df['ciudad'].mode()[0]
moda_genero = df['genero'].mode()[0]

df['ciudad'] = df['ciudad'].fillna(moda_ciudad)
df['genero'] = df['genero'].fillna(moda_genero)

print("Nulos restantes tras la imputación:")
print(df[['edad', 'salario', 'ciudad', 'genero']].isnull().sum())

## asegurar tipos de datos correctos

In [9]:
df['edad'] = pd.to_numeric(df['edad'])
df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')

df.dtypes

nombre                 str
edad               float64
salario            float64
ciudad                 str
genero                 str
nivel                  str
fecha       datetime64[us]
ingresos           float64
gastos             float64
codigo                 str
dtype: object

## filtrar datos fuera del rango

In [10]:
df = df[(df['edad'] >= 0) & (df['edad'] <= 120)]
df = df[df['salario'] <= 1_000_000_000]

print(f"Registros restantes tras filtrar rangos: {len(df)}")

Registros restantes tras filtrar rangos: 28


## convertir variables ordinales

In [11]:
def mapear_nivel(val):
    if pd.isna(val):
        return np.nan
    v = str(val).lower().strip()
    if v == 'bajo':
        return 1
    elif v == 'medio':
        return 2
    elif v == 'alto':
        return 3
    return np.nan

df['nivel'] = df['nivel'].apply(mapear_nivel)
df['nivel'].value_counts(dropna=False)

nivel
NaN    9
1.0    8
2.0    7
3.0    4
Name: count, dtype: int64

## codificacion de variables categoricas

In [12]:
df = pd.get_dummies(df, columns=['genero', 'ciudad'], drop_first=True, dtype=int)
df.head(2)

,nombre,edad,salario,nivel,fecha,ingresos,gastos,codigo,genero_Masculino,genero_Otro,ciudad_Cali,ciudad_Medellín
3,Maria,11.0,4.718402e+03,2.0,2022-10-11,NaN,NaN,XYZ98765,0,0,0,0
4,Laura,94.0,7.477876e+07,3.0,2017-07-11,3102.167244,7570.107277,NaN,1,0,1,0


## escalado de variables numericas

In [13]:
scaler = MinMaxScaler()
df[['edad', 'ingresos']] = scaler.fit_transform(df[['edad', 'ingresos']])
df[['edad', 'ingresos']].describe()

,edad,ingresos
count,28.000000,14.000000
mean,0.456767,0.517285
std,0.329073,0.308373
min,0.000000,0.000000
25%,0.179825,0.302633
50%,0.442982,0.530391
75%,0.796053,0.693090
max,1.000000,1.000000


## eliminar duplicados, crear nueva variable y exportar

In [14]:
# 11. Eliminar duplicados
df = df.drop_duplicates()

# 12. Creación de la columna utilidad
df['utilidad'] = df['ingresos'] - df['gastos']

# 13. Exportación final
df.to_csv('datos_limpios.csv', index=False)
print("¡Dataset exportado exitosamente como 'datos_limpios.csv'!")

# Muestra del resultado final
display(df.head())

¡Dataset exportado exitosamente como 'datos_limpios.csv'!


,nombre,edad,salario,nivel,fecha,ingresos,gastos,codigo,genero_Masculino,genero_Otro,ciudad_Cali,ciudad_Medellín,utilidad
3,Maria,0.096491,4.718402e+03,2.0,2022-10-11,NaN,NaN,XYZ98765,0,0,0,0,NaN
4,Laura,0.824561,7.477876e+07,3.0,2017-07-11,0.193849,7570.107277,NaN,1,0,1,0,-7569.913428
14,Ana,1.000000,3.710838e+08,NaN,2023-12-31,NaN,NaN,ABC12345,0,1,0,0,NaN
20,Laura,0.500000,5.837374e+08,NaN,2012-09-21,NaN,1684.007730,LMN24680,1,0,0,1,NaN
24,Maria,0.605263,4.485451e+03,NaN,2013-11-23,NaN,NaN,LMN24680,0,0,1,0,NaN


In [3]:
%pip install pandas numpy scikit-learn

  Using cached numpy-2.5.4-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/10.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.9 MB ? eta -:--:--
    --------------------------------------- 0.3/10.9 MB ? eta -:--:--
   - -------------------------------------- 0.5/10.9 MB 1.2 MB/s eta 0:00:09
   -- ------------------------------------- 0.8/10.9 MB 1.2 MB/s eta 0:00:09
   --- ------------------------------------ 1.0/10.9 MB 1.1 MB/s eta 0:00:10
   ---- ----------------------------------- 1.3/10.9 MB 1.1 MB/s eta 0:00:09
   ----- ---------------------------------- 1.6/10.9 MB 1.1 MB/s eta 0:00:09
   ------ --------------------------------- 1.8/10.9 MB 1.1 MB/s eta 0:00:09
   ------ --------------------------------- 1.8/10.9 MB 1.1 MB/s eta 0:00:09
   ------ --------------------------------- 1.8/10.9 MB 1.1 MB/s eta 0:00:09
   ------ --------------------------------- 1.8/10.9 MB 1.1 MB/s eta 0:00:09
   ------- ---------------


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
